# 00 · 环境、ComfyUI 与模型

目标：一次跑完 G0–G2，确认 GPU / ROCm / torch / ComfyUI / 模型都对。

顺序：预检 → 安装 → 下载 Set A → 启动 → 核对模型与工作流。

> 每一步失败就停下来修，不要继续往下跑。100 小时额度经不起“先跑起来再说”。

In [ ]:
import os, pathlib
WORK = pathlib.Path(os.environ.get("WORK", "/mnt/workspace/ai-comic-drama"))
os.chdir(WORK)
print("cwd:", pathlib.Path.cwd())
print("exists:", WORK.exists())

## G0 · 预检

通过标准：`torch.cuda.is_available() == True`、设备是 MI300X、显存约 192GB、
`/mnt/workspace` 可用空间 > 95GB、`/dev/kfd` 存在。

In [ ]:
!bash scripts/00_preflight.sh

## G1 · 安装 ComfyUI

脚本会创建 venv、clone ComfyUI、安装最小自定义节点。
**不会**替换镜像自带的 ROCm torch；安装 requirements 时显式排除 torch 三件套。

In [ ]:
!bash scripts/01_bootstrap.sh

In [ ]:
import os
# Jupyter's kernel Python is the system one; make every following !python use the venv.
os.environ["VIRTUAL_ENV"] = str(WORK / "venv")
os.environ["PATH"] = str(WORK / "venv" / "bin") + os.pathsep + os.environ["PATH"]
print("python ->", os.popen("python -c 'import sys; print(sys.executable)'").read().strip())

In [ ]:
!venv/bin/python -c "import torch; print(torch.__version__, torch.version.hip, torch.cuda.is_available(), torch.cuda.get_device_name(0))"

## G2 · 下载模型（Set A）

模型本体约 84.3GB，加上代码 / 缓存 / 输出后总占用约 92GB：图像编辑 2511 + Qwen-Image T2I + Wan2.2 5B + TTS + Qwen3-4B + ASR。

- T2I 只用于生成角色三视图和场景主视觉；资产确认后可以删掉它腾出 20.4GB。
- 14B 正片模型不要现在下，等 G6 闭环通过后再下。
- 下载是断点续传的，失败直接重跑同一条命令。

In [ ]:
!python scripts/02_download_models.py --groups image,image-t2i,video-5b,tts,llm-small,asr

In [ ]:
!python scripts/02_download_models.py --groups image,image-t2i,video-5b,tts,llm-small,asr --verify

In [ ]:
!df -h /mnt/workspace
!du -sh models cache 2>/dev/null || true

## 启动 ComfyUI

默认端口 8188，只监听本机时批处理最稳；要在浏览器看 UI 再去 Notebook 页面做端口映射。

In [ ]:
!bash scripts/03_start_comfyui.sh

In [ ]:
import json, urllib.request
stats = json.load(urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=10))
print(json.dumps(stats.get("devices"), ensure_ascii=False, indent=2))
print("comfy:", stats.get("system", {}).get("comfyui_version"))

## 工作流标题自检

先从 ComfyUI 模板导出 API 工作流到 `workflows/`（见 [workflows/README.md](../workflows/README.md)），
再运行这个单元。没有文件时只打印提示，不报错。

In [ ]:
import json, pathlib
from pipeline.comfy_client import ComfyClient

files = sorted(pathlib.Path("workflows").glob("*.api.json"))
if not files:
    print("还没有 API 工作流。先按 workflows/README.md 导出。")
for p in files:
    wf = json.loads(p.read_text(encoding="utf-8"))
    titles = [(t, c) for t, c, _ in ComfyClient.titles(wf) if t]
    print(p.name, "->", titles)

## Gate 记录

- G0 通过时间：____
- G1 ComfyUI 启动耗时：____ 秒
- G2 模型总占用：____ GB，剩余磁盘：____ GB
- 备注：____

把结果写进 [docs/CHECKLIST.md](../docs/CHECKLIST.md)，然后打开 `01_episode.ipynb`。